In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
pip install transformers datasets sentencepiece 


Note: you may need to restart the kernel to use updated packages.


In [4]:
from datasets import load_dataset

dataset = load_dataset("thainq107/iwslt2015-en-vi")

train_ds = dataset["train"].select(range(100000))
val_ds = dataset["validation"].select(range(1268)) 
print(dataset) 

README.md:   0%|          | 0.00/522 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/181k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/133317 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1268 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1268 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['en', 'vi'],
        num_rows: 133317
    })
    validation: Dataset({
        features: ['en', 'vi'],
        num_rows: 1268
    })
    test: Dataset({
        features: ['en', 'vi'],
        num_rows: 1268
    })
})


In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "Helsinki-NLP/opus-mt-en-vi"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)  

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/809k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/756k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/289M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/289M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [6]:
max_length = 128

def preprocess(example):
    inputs = [
        f"Translate English to Vietnamese:\n{en}"
        for en in example["en"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=max_length,
        truncation=True
    )

    labels = tokenizer(
        example["vi"],
        max_length=max_length,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs 

In [7]:
train_ds = train_ds.map(preprocess, batched=True, remove_columns=["en", "vi"])
val_ds = val_ds.map(preprocess, batched=True, remove_columns=["en", "vi"]) 

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1268 [00:00<?, ? examples/s]

In [8]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model) 

In [9]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./envi_llm",

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=3e-5,
    num_train_epochs=5,

    fp16=True,

    predict_with_generate=True,  
    generation_max_length=128,
    generation_num_beams=5,      

    logging_steps=200,
    save_steps=1000,
    save_total_limit=2,

    report_to="none"
) 

In [16]:
!pip install evaluate sacrebleu
import evaluate

bleu = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = [[(l if l != -100 else tokenizer.pad_token_id) for l in label] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [[l.strip()] for l in decoded_labels]

    result = bleu.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.0 MB/s eta 0:00:00


In [18]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics
) 

In [19]:
trainer.train() 

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
200,6.540476
400,4.832638
600,4.394317
800,4.056342
1000,3.826876
1200,3.665176
1400,3.520241
1600,3.355513
1800,3.273384
2000,3.185063


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=31250, training_loss=2.045832162597656, metrics={'train_runtime': 8813.2414, 'train_samples_per_second': 56.733, 'train_steps_per_second': 3.546, 'total_flos': 9598487283892224.0, 'train_loss': 2.045832162597656, 'epoch': 5.0})

In [20]:
text = "Translate English to Vietnamese:\nI love you"

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_length=128,
    num_beams=5,
    temperature=0.7
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True)) 

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Tôi yêu các bạn


In [24]:
sentences = [
    "Hello!",
    "How are you?",
    "I am a student.",
    "This is a book.",
    "Nice to meet you."
]

for text in sentences:
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=60,
        num_beams=5,
        no_repeat_ngram_size=3
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("EN:", text)
    print("VI:", result)
    print("-" * 50) 

EN: Hello!
VI: Xin chào !
--------------------------------------------------
EN: How are you?
VI: Cậu thế nào rồi ?
--------------------------------------------------
EN: I am a student.
VI: Tôi là một học sinh .
--------------------------------------------------
EN: This is a book.
VI: Đây là một cuốn sách .
--------------------------------------------------
EN: Nice to meet you.
VI: Rất vui được gặp bạn .
--------------------------------------------------


In [25]:
sentences = [
    "Although it was raining, we decided to go out.",
    "If I had known earlier, I would have told you.",
    "The book that you gave me yesterday is very interesting.",
    "Even though he was tired, he continued working.",
    "She said that she would come if she had time."
]

for text in sentences:
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=60,
        num_beams=5,
        no_repeat_ngram_size=3
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("EN:", text)
    print("VI:", result)
    print("-" * 50)  


EN: Although it was raining, we decided to go out.
VI: Mặc dù đang mưa , chúng tôi đã quyết định đi ra ngoài .
--------------------------------------------------
EN: If I had known earlier, I would have told you.
VI: Nếu tôi biết trước đó , tôt có thể nói với bạn .
--------------------------------------------------
EN: The book that you gave me yesterday is very interesting.
VI: Cuốn sách mà bạn đưa cho tôi hôm qua rất thú vị .
--------------------------------------------------
EN: Even though he was tired, he continued working.
VI: mặc dù không khó khăn , anh ấy vẫn tiếp tục làm việc .
--------------------------------------------------
EN: She said that she would come if she had time.
VI: Cô nói rằng cô sẽ đến nếu cô ấy có .
--------------------------------------------------


In [30]:
sentences = [
    "The meeting has been postponed until next week.",
    "Please make sure to submit your assignment on time.",
    "This application helps users translate text quickly.",
    "We are developing a machine translation system.",
    "The results of this experiment are very promising."
]



for text in sentences:
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=60,
        num_beams=5,
        no_repeat_ngram_size=3
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("EN:", text)
    print("VI:", result)
    print("-" * 50) 

EN: The meeting has been postponed until next week.
VI: Cuộc họp đã được tập trung cho đến tuần sau .
--------------------------------------------------
EN: Please make sure to submit your assignment on time.
VI: Hãy chắc chắn làm việc của mình đúng thời gian .
--------------------------------------------------
EN: This application helps users translate text quickly.
VI: ứng dụng này giúp người dịch văn bản nhanh chóng .
--------------------------------------------------
EN: We are developing a machine translation system.
VI: Chúng tôi đang phát triển một hệ thống dịch máy .
--------------------------------------------------
EN: The results of this experiment are very promising.
VI: Kết quả của cuộc thử nghiệm này rất hứa hẹn .
--------------------------------------------------


In [33]:
raw_val = dataset["validation"].select(range(1000))

predictions = []
references = []

model.eval()

for example in raw_val:
    input_text = example["en"]

    # tokenize
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True).to(model.device)

    # generate
    outputs = model.generate(
        **inputs,
        max_length=100,
        num_beams=5,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

    # decode
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    ref = example["vi"]

    predictions.append(pred)
    references.append([ref])  

# tính BLEU
result = bleu.compute(predictions=predictions, references=references)

print("BLEU:", result["score"])  

That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


BLEU: 23.68810806079603


In [34]:
model.save_pretrained("/kaggle/working/LLMversion2-model")
tokenizer.save_pretrained("/kaggle/working/LLMversion2-model")   

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/LLMversion2-model/tokenizer_config.json',
 '/kaggle/working/LLMversion2-model/vocab.json',
 '/kaggle/working/LLMversion2-model/source.spm',
 '/kaggle/working/LLMversion2-model/target.spm',
 '/kaggle/working/LLMversion2-model/added_tokens.json')

In [37]:
!zip -r LLM-modelversion2.zip /kaggle/working/LLMversion2-model   

  adding: kaggle/working/LLMversion2-model/ (stored 0%)
  adding: kaggle/working/LLMversion2-model/target.spm (deflated 50%)
  adding: kaggle/working/LLMversion2-model/source.spm (deflated 51%)
  adding: kaggle/working/LLMversion2-model/generation_config.json (deflated 43%)
  adding: kaggle/working/LLMversion2-model/tokenizer_config.json (deflated 67%)
  adding: kaggle/working/LLMversion2-model/config.json (deflated 64%)
  adding: kaggle/working/LLMversion2-model/vocab.json (deflated 70%)
  adding: kaggle/working/LLMversion2-model/model.safetensors (deflated 7%)
